# Publication experiments — Voronoi–lattice compression

This notebook reproduces every table of the experimental section. Place it at the repository root, next to `src/` and `article_assets/`.

Each table is rendered as a 300 dpi PNG image into `article_assets/`, and each cell prints the matching `\includegraphics` snippet for `main.tex`.

**Protocol (aligned with Göttlich et al. 2025):** N = 3000, Gaussian noise N(0, σ = 0.02) (σ is the standard deviation), errors reported as the average over `N_REPEATS` replicates of the per-replicate deviation |err − app_L|. Set `N_REPEATS = 100` for the final run (10 for exploration). Each computing cell has its own `R_LOCAL = N_REPEATS` line at the top so a single cell can be re-run quickly without changing the global protocol.

**Methods:** `V+Lat` — Voronoi assignment on a CBC rank-1 lattice (L the nearest prime to the dyadic target); `V+Sob` — Voronoi assignment on an unscrambled Sobol' net; `SC` — standard supercompress (Joseph & Mak 2021, split on X by response loss); `Avg+Sob` — Dick–Feischl averaging (ν = m/2); `GHW+Lat` — Gnewuch–Harsha–Wnuk lattice compressor (step-hyperbolic-cross weights via the Dirichlet-kernel factorisation; uses `src/index_sets.py` and `src/ghw_fast.py`).

**f2:** the reference definition (exponential branch on the corner {x1 ≤ u1} ∧ {x2 ≤ u2}) is used throughout; raw f2 values are therefore not directly comparable with the public code of Göttlich et al., whose implementation uses the disjunction (see the methodological remarks in the paper).


### Cell 1 — Configuration & high-resolution rendering infrastructure
Defines the experimental protocol (N, σ, replications, seed, unscrambled Sobol), all sweep parameters in one place, the matplotlib settings for publication-quality output, and the two rendering helpers: `save_fig` (figures) and `render_table` (booktabs-style **table images**, vector PDF + 300 dpi PNG, with the ready-to-paste `\includegraphics` snippet). Also caches CBC lattice constructions so they are built once per (L, d).


In [ ]:
# %% Cell 1 — configuration, imports, high-resolution rendering setup
import sys, os, time, math, re
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from src.lattices import cbc_construction, lattice_points
from src.digital_nets import sobol_points
from src.compressors import VoronoiQMCCompressor, SupercompressFaithful
from src.qmc_averaging import compute_dick_feischl_weights, approximate_loss_averaging
from src.ghw_lattice import compute_ghw_weights_step_hc, approximate_loss_ghw
try:    # optimised drop-in (identical output, ~5-10x faster in high d)
    from src.ghw_fast import compute_ghw_weights_step_hc_fast \
        as compute_ghw_weights_step_hc
except ImportError:
    pass  # falls back to the reference implementation
from src.test_functions import get_test_function, DISTINCT_FORMS

# ----------------------------- protocol ------------------------------------
N_REPEATS   = 10          # <<< set 100 for the final article run
N_POINTS    = 3000
NOISE_STD   = 0.02
SEED        = 12345
SCRAMBLE    = False       # unscrambled Sobol, as in Gottlich's QMCVoronoi.m

# sweep parameters (reduce for a quick check)
L_TARGETS_T1 = [32, 128, 256, 512, 1024]
RATES_T4     = [128, 256, 512]
DIMS_T3      = [2, 3, 5, 10]
L_T3         = 256
S_LIST_T5    = [2, 3, 5]
K_LIST_T5    = [2**6, 2**8, 2**10, 2**12]
N_T5         = 10000
R_TIME       = 3
FORMS_ALL    = ["f1", "f2", "f3", "genz_oscillatory", "genz_product_peak",
                "genz_corner_peak", "genz_gaussian"]   # the 7 distinct forms
L_FORMS_TARGET = 64   # theory-optimal size: L* = N^{d/(d+2)} = sqrt(3000) = 55
                      # (Corollary 'optimal regime'); nearest dyadic 2^6 = 64,
                      # nearest prime 67 for the lattice/SC methods

OUTDIR = "article_assets"
os.makedirs(OUTDIR, exist_ok=True)

matplotlib.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.family": "serif", "font.size": 10, "mathtext.fontset": "cm",
    "axes.labelsize": 10, "axes.titlesize": 10,
    "legend.fontsize": 8.5, "legend.frameon": False,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "lines.linewidth": 1.4, "lines.markersize": 4.5,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.5,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})

def _snippet(name, caption):
    print("LaTeX:\n"
          "\\begin{table}[htbp]\\centering\n" if name.startswith("table") else
          "LaTeX:\n\\begin{figure}[htbp]\\centering\n", end="")
    print(f"\\includegraphics[width=.9\\textwidth]{{{name}.png}}")
    env = "table" if name.startswith("table") else "figure"
    print(f"\\caption{{{caption}}}\\label{{{'tab' if env=='table' else 'fig'}:{name}}}")
    print(f"\\end{{{env}}}\n")

def save_fig(fig, name, caption="..."):
    png = os.path.join(OUTDIR, name + ".png")
    fig.savefig(png)
    print(f"saved: {png}")
    _snippet(name, caption)

# ---------------------- booktabs-style TABLE IMAGES -------------------------
def _disp_len(s):
    """approximate display length of a cell (math markup stripped)."""
    t = re.sub(r"\\cdot10\^\{(-?\d+)\}", "x10-0", s)
    t = re.sub(r"[\$\{\}\^_]|\\[a-zA-Z]+", "", t)
    return max(len(t), 2)

def render_table(name, header, rows, caption="...", fontsize=10):
    """Render a booktabs-style table as a high-resolution image (PDF+PNG)."""
    ncol = len(header)
    data = [header] + [[str(x) for x in r] for r in rows]
    widths = [max(_disp_len(row[j]) for row in data) for j in range(ncol)]
    char_w = 0.085 * (fontsize / 10)          # inches per character
    col_w  = [max(w * char_w + 0.28, 0.55) for w in widths]
    xs = np.concatenate([[0], np.cumsum(col_w)])
    W = xs[-1]
    row_h = 0.30 * (fontsize / 10)
    H = row_h * (len(data) + 1.0)
    fig, ax = plt.subplots(figsize=(W, H))
    ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis("off"); ax.grid(False)
    y_top = H - 0.5 * row_h
    ys = [y_top - (i + 0.5) * row_h for i in range(len(data))]
    for i, row in enumerate(data):
        for j, cell in enumerate(row):
            if j == 0:                       # first column left-aligned
                x, ha = xs[j] + 0.10, "left"
            else:                            # others centred
                x, ha = 0.5 * (xs[j] + xs[j + 1]), "center"
            ax.text(x, ys[i], cell, ha=ha, va="center", fontsize=fontsize,
                    fontweight="bold" if i == 0 else "normal")
    lw_heavy, lw_light = 1.3, 0.7
    ax.hlines(y_top, 0, W, lw=lw_heavy, color="k")                     # toprule
    ax.hlines(y_top - row_h, 0, W, lw=lw_light, color="k")             # midrule
    ax.hlines(y_top - len(data) * row_h, 0, W, lw=lw_heavy, color="k") # bottomrule
    png = os.path.join(OUTDIR, name + ".png")
    fig.savefig(png); plt.show()
    print(f"saved: {png}")
    _snippet(name, caption)

# ------------------------------ helpers -------------------------------------
def next_prime(n):
    def isp(m):
        if m < 2: return False
        for i in range(2, int(m ** 0.5) + 1):
            if m % i == 0: return False
        return True
    while not isp(n): n += 1
    return n

def sci(v):
    """1.234e-05 -> $1.23\\cdot10^{-5}$ (rendered by mathtext in the image)."""
    if v is None or (isinstance(v, float) and np.isnan(v)): return "--"
    m, e = f"{v:.2e}".split("e")
    return f"${m}\\cdot10^{{{int(e)}}}$"

_LATTICE_CACHE = {}
def get_lattice(L_prime, d):
    key = (L_prime, d)
    if key not in _LATTICE_CACHE:
        t0 = time.perf_counter()
        g = cbc_construction(L_prime, d, alpha=2)
        Z = lattice_points(g, L_prime)
        _LATTICE_CACHE[key] = (Z, time.perf_counter() - t0)
    return _LATTICE_CACHE[key]

def df_feasible(m, d, nu):
    return math.comb(nu + d - 1, d - 1) <= 5000

# --- GHW (Gnewuch--Harsha--Wnuk) step-hyperbolic-cross parameters ----------
GHW_ALPHA = 2
def ghw_gamma(d):
    return np.array([1.0 / (j + 1) ** 2 for j in range(d)])  # same product
                                                             # weights as CBC
def ghw_m(L, d):
    # calibration of GHW Cor. 3.20: m ~ (alpha - 1/2) log2 L + (d-1)/2
    return int(round((GHW_ALPHA - 0.5) * np.log2(L) + (d - 1) / 2))
_GHW_BLOCKS = {}
def ghw_feasible(d, m):
    # cost is driven by the RETAINED blocks of the step-HC decomposition
    # (most shells are empty under decaying weights); with the fast
    # implementation, up to ~500 blocks (d = 10, m = 17: ~20 s/call)
    # remain tractable.
    key = (d, m)
    if key not in _GHW_BLOCKS:
        from src.index_sets import step_hyperbolic_cross_decomposition
        _GHW_BLOCKS[key] = len(
            step_hyperbolic_cross_decomposition(m, ghw_gamma(d), GHW_ALPHA))
    return _GHW_BLOCKS[key] <= 500

print("config OK — N_REPEATS =", N_REPEATS)


### Cell 2 — Benchmark engine
`bench(f, d, L_target, ...)`: for each replicate, draws X ~ U[0,1)^d and Y = f(X) + N(0,σ²), computes the exact loss err(f), runs each method (V+Lat on the nearest prime ≥ L; V+Sob and Avg+Sob on 2^m; SC = **standard** Joseph–Mak supercompress on the prime), and records the per-replicate deviation |err − app_L| and the wall-clock fit time. Returns means and standard deviations. The Dick–Feischl weights are guarded by a combinatorial cost bound and use ν = m/2, as in Göttlich's code.


In [ ]:
# %% Cell 2 — benchmark engine
def bench(f, d, L_target, N=N_POINTS, R=N_REPEATS, noise=NOISE_STD,
          seed=SEED, methods=("V+Lat", "V+Sob", "SC", "Avg+Sob", "GHW+Lat")):
    """{method: dict(err, err_std, time_ms)}; err = mean_R |err_true - app_L|."""
    rng = np.random.default_rng(seed)
    Lp = next_prime(L_target)
    m  = int(round(np.log2(L_target))); L2 = 2 ** m
    nu = m // 2
    acc = {mth: [] for mth in methods}
    tim = {mth: [] for mth in methods}
    Zl, _ = get_lattice(Lp, d) if ("V+Lat" in methods or "GHW+Lat" in methods) \
            else (None, 0.0)
    Ps = sobol_points(L=L2, d=d, seed=0, scramble=SCRAMBLE) \
         if ("V+Sob" in methods or "Avg+Sob" in methods) else None
    for rep in range(R):
        X = rng.uniform(0, 1, size=(N, d))
        Y = f(X) + noise * rng.standard_normal(N)
        et = float(np.mean((f(X) - Y) ** 2))
        Ysq = float(np.mean(Y ** 2))
        if "V+Lat" in methods:
            t0 = time.perf_counter()
            c = VoronoiQMCCompressor(Zl); Zc, Wc = c.fit(X, Y)
            e = abs(et - c.approximate_loss(f, Zc, Wc))
            acc["V+Lat"].append(e); tim["V+Lat"].append(time.perf_counter() - t0)
        if "V+Sob" in methods:
            t0 = time.perf_counter()
            c = VoronoiQMCCompressor(Ps); Zc, Wc = c.fit(X, Y)
            e = abs(et - c.approximate_loss(f, Zc, Wc))
            acc["V+Sob"].append(e); tim["V+Sob"].append(time.perf_counter() - t0)
        if "SC" in methods:
            t0 = time.perf_counter()
            sc = SupercompressFaithful(L=Lp, lam=0.0, random_state=rep)
            Zc, Wc = sc.fit(X, Y)
            e = abs(et - sc.approximate_loss(f, Zc, Wc))
            acc["SC"].append(e); tim["SC"].append(time.perf_counter() - t0)
        if "GHW+Lat" in methods:
            mg = ghw_m(Lp, d)
            if ghw_feasible(d, mg):
                t0 = time.perf_counter()
                W1g, W2g = compute_ghw_weights_step_hc(
                    X, Y, Zl, m=mg, gamma=ghw_gamma(d), alpha=GHW_ALPHA)
                e = abs(et - approximate_loss_ghw(f, Zl, W1g, W2g, Ysq))
                acc["GHW+Lat"].append(e)
                tim["GHW+Lat"].append(time.perf_counter() - t0)
        if "Avg+Sob" in methods and df_feasible(m, d, nu) and nu >= 1:
            t0 = time.perf_counter()
            W1, W2 = compute_dick_feischl_weights(X, Y, Ps, nu=nu, m=m, b=2)
            e = abs(et - approximate_loss_averaging(f, Ps, W1, W2, Ysq))
            acc["Avg+Sob"].append(e); tim["Avg+Sob"].append(time.perf_counter() - t0)
    out = {}
    for mth in methods:
        if acc[mth]:
            out[mth] = dict(err=float(np.mean(acc[mth])),
                            err_std=float(np.std(acc[mth])),
                            time_ms=1000 * float(np.mean(tim[mth])))
        else:
            out[mth] = dict(err=float("nan"), err_std=float("nan"),
                            time_ms=float("nan"))
    return out, Lp, L2

print("engine OK")


### Cell 3 — Table 1 image: benchmark vs compression rate (analog of Göttlich Table 1)
Average error for f1, f2, f3 at d=2 across rates 1–34% (L = 32…1024), for all five methods — Göttlich's Table 1 extended with **GHW+Lat** (Gnewuch–Harsha–Wnuk, step-hyperbolic-cross weights, α=2, m per their Cor. 3.20) and the two Voronoi-QMC columns. This is the table where our f1/f3 columns are directly comparable with theirs (same order verified) and where f2 is *not* (reference definition — see the paper's methodological remark 3).


In [ ]:
# %% Cell 3 — TABLE 1 image: average error, d=2, f1/f2/f3, all methods
R_LOCAL = N_REPEATS   # <-- per-cell override: set e.g. R_LOCAL = 10 to run
                      #     just this cell quickly, without touching the global
d = 2
T1 = {}   # reused by Figure 1
rows = []
for L_t in L_TARGETS_T1:
    for k, f_name in enumerate(["f1", "f2", "f3"]):
        f = get_test_function(f_name)
        res, Lp, L2 = bench(f, d, L_t, R=R_LOCAL)
        T1.setdefault(f_name, {})[L_t] = res
        rate = f"{100*Lp/N_POINTS:.0f} %" if k == 0 else ""
        rows.append([rate, f"{L_t}" if k == 0 else "", f"$f_{f_name[-1]}$",
                     sci(res["SC"]["err"]), sci(res["Avg+Sob"]["err"]),
                     sci(res["GHW+Lat"]["err"]),
                     sci(res["V+Sob"]["err"]), sci(res["V+Lat"]["err"])])
render_table(
    "table1",
    ["rate", "$L$", "$f$", "SC", "Avg+Sob", "GHW+Lat", "V+Sob", "V+Lat"],
    rows,
    caption=(f"Average compression error, $d=2$, $N={N_POINTS}$, "
             f"$\\sigma={NOISE_STD}$, {R_LOCAL} replications. Lattice "
             "methods use the nearest prime $\\ge L$; net methods use "
             "$L=2^m$. $f_2$: reference definition."))


### Cell 4 — MAIN TABLE image (`table_forms`): all methods × the 7 distinct functional forms
The centrepiece for the claim “our method is better in more settings”. Fixed d = 2 and **L chosen by the theory, not by us**: the optimal-regime corollary gives L* ≍ N^{d/(d+2)} = √3000 ≈ 55, hence L = 2⁶ = 64 (nets) / 67 (lattices). Columns: SC, Avg+Sob, GHW+Lat, V+Sob, V+Lat, and the gain factor V+Sob/V+Lat. Expected outcome (verified in exploration): V+Lat dominates V+Sob on all 7 forms; against SC the picture is form-dependent, which the paper discusses honestly.


In [ ]:
# %% Cell 4 — MAIN TABLE (table_forms): all 4 methods on the 7 distinct forms
R_LOCAL = N_REPEATS   # <-- per-cell override: set e.g. R_LOCAL = 10 to run
                      #     just this cell quickly, without touching the global
# d = 2, L = L_FORMS_TARGET, chosen by the optimal-regime corollary
# (L* = N^{d/(d+2)} = sqrt(N) ~ 55 for N = 3000 -> dyadic 64, prime 67).
d_forms = 2
RES_FORMS = {}
rows = []
pretty = {"f1": "$f_1$ (continuous exp.)", "f2": "$f_2$ (discontinuous)",
          "f3": "$f_3$ (Zhou)", "genz_oscillatory": "$g_1$ (oscillatory)",
          "genz_product_peak": "$g_2$ (product peak)",
          "genz_corner_peak": "$g_3$ (corner peak)",
          "genz_gaussian": "$g_4$ (Gaussian)"}
for f_name in FORMS_ALL:
    f = get_test_function(f_name)
    res, Lp, L2 = bench(f, d_forms, L_FORMS_TARGET, R=R_LOCAL)
    RES_FORMS[f_name] = res
    gain = res["V+Sob"]["err"] / res["V+Lat"]["err"]
    rows.append([pretty[f_name],
                 sci(res["SC"]["err"]), sci(res["Avg+Sob"]["err"]),
                 sci(res["GHW+Lat"]["err"]),
                 sci(res["V+Sob"]["err"]), sci(res["V+Lat"]["err"]),
                 f"${gain:.1f}\\times$"])
    print(f"{f_name:22s} SC={res['SC']['err']:.2e} Avg={res['Avg+Sob']['err']:.2e} "
          f"VSob={res['V+Sob']['err']:.2e} VLat={res['V+Lat']['err']:.2e} "
          f"gain={gain:.1f}x")
render_table(
    "table_forms",
    ["functional form", "SC", "Avg+Sob", "GHW+Lat", "V+Sob", "V+Lat",
     "gain V+Sob/V+Lat"],
    rows,
    caption=(f"All four methods on the seven distinct functional forms, "
             f"$d=2$, $N={N_POINTS}$, $\\sigma={NOISE_STD}$, {N_REPEATS} "
             f"replications, at the theory-prescribed compression size "
             f"$L^*\\approx N^{{d/(d+2)}}=\\sqrt{{N}}\\approx55$ "
             f"($L=2^6$ for net methods, $L=67$ for lattice methods)."))


### Cell 5 — Table robustness (`table_robustness`): sensitivity to the difficulty parameters
Probes whether the V+Lat-vs-V+Sob comparison depends on the parametrisation, using the two duplicated forms (continuous exponential as f₁/a=5 and g₅/Genz-flat; discontinuous as f₂ and g₆) at two sizes (L* and the 9% rate), with **raw errors and gains**. Expected outcome (100 reps): the gain persists for the discontinuous form at both levels and both sizes, and for a=5 at both sizes; in the **flat regime (g₅) at small L it reverses** — both methods sit at the stochastic floor (~10⁻⁴) where cell geometry is immaterial — and re-establishes at L=257. The paper's Section 4.3 reports exactly this reading.


In [ ]:
# %% Cell 5 — TABLE robustness (`table_robustness`): duplicated forms,
# two parameter levels, two sizes (L* and the 9% rate).
R_LOCAL = N_REPEATS   # <-- per-cell override (e.g. R_LOCAL = 10)
label = {"f1": ("continuous exp.", "$a_j=5$ ($f_1$)"),
         "genz_continuous": ("continuous exp.", "Genz ($g_5$)"),
         "f2": ("discontinuous", "$a_j=5$ ($f_2$)"),
         "genz_discontinuous": ("discontinuous", "Genz ($g_6$)")}
rows = []
for Lt, Ltxt in [(L_FORMS_TARGET, f"${next_prime(L_FORMS_TARGET)}"
                                  "\\;(\\approx L^*)$"),
                 (256, "$257$")]:
    for k, fn in enumerate(["f1", "genz_continuous", "f2",
                            "genz_discontinuous"]):
        f = get_test_function(fn)
        res, Lp, L2 = bench(f, 2, Lt, R=R_LOCAL,
                            methods=("V+Sob", "V+Lat"))
        vs, vl = res["V+Sob"]["err"], res["V+Lat"]["err"]
        form, level = label[fn]
        rows.append([Ltxt if k == 0 else "", form if k % 2 == 0 else "",
                     level, sci(vs), sci(vl), f"${vs/vl:.1f}\\times$"])
        print(f"L={Lt:4d} {fn:20s} V+Sob={vs:.3e} V+Lat={vl:.3e} "
              f"gain={vs/vl:.2f}x")
render_table(
    "table_robustness",
    ["$L$", "form", "parameters", "V+Sob", "V+Lat", "gain"],
    rows,
    caption=(f"Robustness to the difficulty parameters: raw errors and "
             f"gains for the duplicated forms; $d=2$, $N={N_POINTS}$, "
             f"$\\sigma={NOISE_STD}$, {R_LOCAL} replications, at "
             f"$L\\approx L^*$ and at the 9\\% rate."))


### Cell 6 — Table 3 image: dimension sensitivity (analog of Göttlich Table 3)
All five methods at the 9% rate (L = 256/257) for d ∈ {2, 3, 5, 10} — **including GHW+Lat in every dimension**. To keep the GHW cost acceptable (~3.7 s/replicate at d=5, ~20 s at d=10 via the fast cached-kernel implementation in `src/ghw_fast.py`), this cell runs at **`R_LOCAL = 30` replications, as a stated exception** to the 100-replication protocol; the caption declares it. Estimated total runtime ≈ 45–50 min (d=10 dominates, ~32 min). A per-configuration progress line is printed so you can follow the run. Note: f2 has enormous replicate variance in d ≥ 3 (its magnitude grows like e^{5d}); read that column qualitatively. Requires `src/ghw_fast.py` and `src/index_sets.py` (both provided).


In [ ]:
# %% Cell 6 — TABLE 3 image: dimension sensitivity (analog of Göttlich Table 3)
# EXCEPTION TO THE GLOBAL PROTOCOL: R_LOCAL = 30 for this cell, so that the
# GHW+Lat column can be computed in ALL dimensions, including d = 10, at an
# acceptable cost (~20 s per replicate at d = 10 with src/ghw_fast.py).
# Estimated total runtime at R_LOCAL = 30:  d=2,3 a few minutes;
# d=5 ~ 6 min (GHW ~3.7 s/rep); d=10 ~ 32 min (GHW ~20 s/rep, 3 functions).
R_LOCAL = 30          # <-- exception (global elsewhere: N_REPEATS = 100)
rows = []
for dd in DIMS_T3:
    for k, f_name in enumerate(["f1", "f2", "f3"]):
        f = get_test_function(f_name)
        t0 = time.perf_counter()
        res, Lp, L2 = bench(f, dd, L_T3, R=R_LOCAL)
        print(f"d={dd} {f_name}: done in {time.perf_counter()-t0:.0f}s  "
              + "  ".join(f"{m}={res[m]['err']:.2e}" for m in res), flush=True)
        rows.append([f"{dd}" if k == 0 else "", f"$f_{f_name[-1]}$",
                     sci(res["SC"]["err"]), sci(res["Avg+Sob"]["err"]),
                     sci(res["GHW+Lat"]["err"]),
                     sci(res["V+Sob"]["err"]), sci(res["V+Lat"]["err"])])
render_table(
    "table3",
    ["$d$", "$f$", "SC", "Avg+Sob", "GHW+Lat", "V+Sob", "V+Lat"],
    rows,
    caption=(f"Average compression error at the 9\\% rate ($L={L_T3}$), "
             f"$N={N_POINTS}$, $\\sigma={NOISE_STD}$, as a function of the "
             f"dimension. {R_LOCAL} replications for this table (exception "
             f"to the {N_REPEATS}-replication protocol, so that "
             "GHW+Lat can be computed in all dimensions including $d=10$; "
             "its cost is $\\approx 20$\\,s per replicate there). "
             "`--' = method skipped by the cost guard."))


In [ ]:
# %% Cell 6bis — TABLE 3* image: dimension study at the THEORY-OPTIMAL size
# Instead of the fixed 9% rate, each dimension uses its own optimal size
#     L*(d) ≍ N^{d/(d+2)}   (Corollary "optimal regime"),
# rounded to the nearest dyadic (net methods) / next prime (lattice methods):
#     d=2 -> 64/67,  d=3 -> 128/131,  d=5 -> 256/257,  d=10 -> 1024/1031.
# Cross-checks: the d=2 row must match the 7-forms table (same L*), and the
# d=5 row must match Table 3 (L*≈305 rounds to the same 256/257).
#
# EXCEPTION TO THE GLOBAL PROTOCOL: R_LOCAL = 30 (as in Table 3), so that
# GHW+Lat can be computed where tractable. At d=10 the optimal size L*=1031
# pushes the GHW level to m=20 with 954 retained blocks (~189 s/replicate,
# ≈ 4.7 h for 30 reps x 3 functions): it is EXCLUDED by default ("--").
# To force it anyway, set GHW_BLOCK_LIMIT = 1000 below and plan ~5 h.
R_LOCAL = 30
#GHW_BLOCK_LIMIT = 500          # <-- set 1000 to force GHW at d=10 (~4.7 h)
GHW_BLOCK_LIMIT =1000

def ghw_feasible(d, m):        # override picked up by bench() (same globals)
    key = (d, m)
    if key not in _GHW_BLOCKS:
        from src.index_sets import step_hyperbolic_cross_decomposition
        _GHW_BLOCKS[key] = len(
            step_hyperbolic_cross_decomposition(m, ghw_gamma(d), GHW_ALPHA))
    return _GHW_BLOCKS[key] <= GHW_BLOCK_LIMIT

rows = []
for dd in DIMS_T3:
    L_star = N_POINTS ** (dd / (dd + 2))                 # theory-optimal size
    L_t = 2 ** int(round(np.log2(L_star)))               # nearest dyadic
    Lp = next_prime(L_t)
    print(f"d={dd}: L* = N^(d/(d+2)) = {L_star:.0f}  ->  "
          f"L = {L_t} (nets) / {Lp} (lattices)", flush=True)
    for k, f_name in enumerate(["f1", "f2", "f3"]):
        f = get_test_function(f_name)
        t0 = time.perf_counter()
        res, Lp, L2 = bench(f, dd, L_t, R=R_LOCAL)
        gain = res["V+Sob"]["err"] / res["V+Lat"]["err"]
        print(f"d={dd} {f_name}: done in {time.perf_counter()-t0:.0f}s  "
              + "  ".join(f"{m}={res[m]['err']:.2e}" for m in res)
              + f"  gain={gain:.1f}x", flush=True)
        rows.append([f"{dd}" if k == 0 else "",
                     f"${Lp}$" if k == 0 else "", f"$f_{f_name[-1]}$",
                     sci(res["SC"]["err"]), sci(res["Avg+Sob"]["err"]),
                     sci(res["GHW+Lat"]["err"]),
                     sci(res["V+Sob"]["err"]), sci(res["V+Lat"]["err"]),
                     f"${gain:.1f}\\times$"])
render_table(
    "table3_Lstar",
    ["$d$", "$L^*$", "$f$", "SC", "Avg+Sob", "GHW+Lat", "V+Sob", "V+Lat",
     "gain V+Sob/V+Lat"],
    rows,
    caption=(f"Average compression error at the theory-optimal size "
             f"$L^*(d)\\approx N^{{d/(d+2)}}$ per dimension "
             f"($N={N_POINTS}$, $\\sigma={NOISE_STD}$, {R_LOCAL} "
             "replications --- stated exception to the "
             f"{N_REPEATS}-replication protocol). Lattice-based methods "
             "use the next prime, net-based methods the nearest dyadic. "
             "`--' = method excluded by the cost guard "
             "(GHW+Lat at $d=10$, $L^*=1031$: $m=20$, $954$ blocks, "
             "$\\approx 4.7$\\,h for this table)."))

### Cell 7 — Timing: per-function compression time (Section 4.6)

Göttlich-style running-time table (cf. their Table 5): methods in rows, test functions in columns, values in seconds. Measured at d = 2 and the theory-optimal size L* = 67 (net L = 64), the same operating point as `table_forms`, so error and cost can be read off the same configuration. The per-data-set cost is driven by X (nearest-node assignment for the Voronoi methods, input-space clustering for supercompress) or by the weight computation over (X, Y) for the weighted schemes, so it is largely independent of the target. The cell prints the LaTeX body of the timing table.


In [ ]:
# %% Cell 7 — TABLE 10: per-function compression time (Section 4.6)
#
# Göttlich-style running-time table (cf. their Table 5): rows = methods,
# columns = test functions, values = average compression (fit) time in
# SECONDS. Measured at d = 2 and the theory-optimal size L* = 67 (net
# L = 64), i.e. the SAME operating point as the error table `table_forms`
# (Table 5 of the article), so a reader can read error and time off the
# same configuration. All FIVE methods are feasible at this size (the
# weighted schemes require K <= 2^8), unlike the large-L regime.
#
# The per-data-set time for the Voronoi methods and supercompress is
# driven by X (nearest-node assignment / input-space clustering), while
# for the weighted schemes it is driven by the inclusion-exclusion
# (resp. Fourier) weight computation over (X, Y); reporting per function
# lets the reader see directly how little the target influences the fit
# time, mirroring the layout of the reference study.
#
# PRINTS the LaTeX body of Table 10 (tab:timing). Copy it into main.tex.
import time
import numpy as np

TIME_D     = 2
TIME_K     = 64                       # net size; lattice uses next prime = 67
TIME_FUNCS = ["f1", "f2", "f3"]
TIME_N     = 10_000
TIME_R     = 10                       # timing repetitions (medians reported)

m_t  = int(np.log2(TIME_K)); nu_t = m_t // 2
Kp_t = next_prime(TIME_K)
Zl_t, t_cbc = get_lattice(Kp_t, TIME_D)
Ps_t = sobol_points(L=TIME_K, d=TIME_D, seed=0, scramble=SCRAMBLE)
mg_t = ghw_m(Kp_t, TIME_D)

def _sec(fn):
    ts = []
    for _ in range(TIME_R):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

# rows in Göttlich order (adaptive, then QMC-averaging, then Voronoi family)
METHOD_ORDER = ["SC", "Avg+Sob", "GHW+Lat", "V+Sob", "V+Lat"]
TIME = {m: {} for m in METHOD_ORDER}

for fname in TIME_FUNCS:
    f = get_test_function(fname)
    rng = np.random.default_rng(SEED + TIME_D * 1000 + TIME_K)
    X = rng.uniform(0, 1, size=(TIME_N, TIME_D)); Y = f(X)
    TIME["V+Lat"][fname]   = _sec(lambda: VoronoiQMCCompressor(Zl_t).fit(X, Y))
    TIME["V+Sob"][fname]   = _sec(lambda: VoronoiQMCCompressor(Ps_t).fit(X, Y))
    TIME["SC"][fname]      = _sec(
        lambda: SupercompressFaithful(L=Kp_t, lam=0.0, random_state=0).fit(X, Y))
    TIME["Avg+Sob"][fname] = _sec(
        lambda: compute_dick_feischl_weights(X, Y, Ps_t, nu=nu_t, m=m_t, b=2))
    TIME["GHW+Lat"][fname] = _sec(
        lambda: compute_ghw_weights_step_hc(X, Y, Zl_t, m=mg_t,
                                            gamma=ghw_gamma(TIME_D),
                                            alpha=GHW_ALPHA))

def _fmt(s):
    return f"{s:.3f}" if s >= 1e-3 else f"{s:.2e}"

print("% --- Table 10 body (copy into tab:timing) ------------------------")
print(f"% d={TIME_D}, L*={Kp_t} (net L={TIME_K}), N={TIME_N}, "
      f"{TIME_R} reps, seconds")
for m in METHOD_ORDER:
    cells = " & ".join(_fmt(TIME[m][fn]) for fn in TIME_FUNCS)
    print(f"\\textsc{{{m}}} & {cells} \\\\")
print("% -----------------------------------------------------------------")
print(f"\nOne-off construction (not in the table): "
      f"CBC lattice = {1000*t_cbc:.0f} ms, "
      f"Sobol' net = negligible, both reused across data sets.")
